# YOLO26s / YOLO26x Fine-Tune — Thermal Human Detection from UAV

Fine-tune model YOLO26 pada dataset Roboflow `thermal-disasters-project/thermal-human-detection-from-uav` v1 (654 frame thermal UAV, 1 kelas `Human`). Notebook ini dikonfigurasi untuk **2× NVIDIA T4** di Kaggle.

## Wajib sebelum Run All
1. Kaggle **Settings → Accelerator → GPU T4 x2**.
2. Kaggle **Add-ons → Secrets → Add a new secret** dengan nama persis `ROBOFLOW_API_KEY`; isi nilainya dengan API key Roboflow milikmu. Notebook membaca secret ini dan **tidak pernah mencetak nilainya**.
3. Pastikan **Internet = On** agar Kaggle dapat mengunduh dataset Roboflow dan bobot YOLO26.

`MODEL_VARIANT = "x"` adalah default. Ubah menjadi `"s"` bila ingin YOLO26s; batch YOLO26x diset lebih kecil agar muat pada dua T4.

Alur: cek GPU → baca secret → download dataset via Roboflow API → validasi split → training multi-GPU → validasi valid/test → export `.pt`, `.onnx`, TensorRT `.engine` → zip artefak.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

import torch

GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT < 2:
    raise RuntimeError(f"Notebook harus memakai GPU T4 x2; GPU terdeteksi: {GPU_COUNT}")

DEVICES = [0, 1]
print("GPU aktif:", [torch.cuda.get_device_name(i) for i in DEVICES])

In [ ]:
!pip install -q --upgrade ultralytics roboflow pyyaml

import ultralytics
import roboflow
print("ultralytics", ultralytics.__version__)
print("roboflow SDK siap")

In [ ]:
import os
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret("ROBOFLOW_API_KEY")
except Exception:
    api_key = None

if not api_key:
    from getpass import getpass
    api_key = getpass("Masukkan API key Roboflow (input tidak ditampilkan): ").strip()
if not api_key:
    raise RuntimeError("API key Roboflow belum tersedia.")

DATA_DIR = Path("/kaggle/working/thermal-human-detection-uav-night")
INPUT_ZIP = next(
    Path("/kaggle/input").glob("*/thermal-human-detection-from-uav*.zip"),
    None,
)
DATASET_VERSION = 1
DATASET_FORMAT = "yolov11"
MODEL_VARIANT = "x"  # pilih "s" atau "x"
MODEL_NAME = f"yolo26{MODEL_VARIANT}.pt"
IMGSZ = 640
EPOCHS = 100
BATCH = {"s": 64, "x": 16}[MODEL_VARIANT]  # total batch; dibagi oleh 2 GPU
RUN_DIR = Path(f"/kaggle/working/run26{MODEL_VARIANT}-thermal")

print("API key tersedia:", bool(api_key))
print({"model": MODEL_NAME, "devices": DEVICES, "batch_total": BATCH, "epochs": EPOCHS})

In [ ]:
if not DATA_DIR.exists():
    if api_key:
        from roboflow import Roboflow
        rf = Roboflow(api_key=api_key)
        project = rf.workspace("thermal-disasters-project").project("thermal-human-detection-from-uav")
        project.version(DATASET_VERSION).download(
            DATASET_FORMAT,
            location=str(DATA_DIR),
            overwrite=True,
        )
    elif INPUT_ZIP:
        import zipfile
        with zipfile.ZipFile(INPUT_ZIP) as z:
            z.extractall(DATA_DIR)
    else:
        raise SystemExit(
            "Tidak ada API key maupun zip - tambahkan Kaggle Secret "
            "ROBOFLOW_API_KEY atau upload zip dataset."
        )
print("dataset siap:", DATA_DIR.exists())

In [ ]:
import yaml

cfg = yaml.safe_load((DATA_DIR / "data.yaml").read_text())
if cfg.get("names") not in (["Human"], {0: "Human"}):
    raise ValueError(f"Dataset harus punya satu kelas Human, mendapat: {cfg.get('names')!r}")

def image_count(split):
    image_dir = DATA_DIR / split / "images"
    return sum(1 for p in image_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"})

counts = {split: image_count(split) for split in ("train", "valid", "test")}
if any(counts[split] == 0 for split in ("train", "valid")):
    raise RuntimeError(f"Split train/valid tidak lengkap: {counts}")
print("kelas:", cfg.get("names"))
print("jumlah gambar:", counts)

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME)
model.train(
    data=str(DATA_DIR / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=20,
    cache=True,
    device=DEVICES,  # multi-GPU DDP: GPU 0 + GPU 1
    workers=4,
    project=str(RUN_DIR),
    name="train",
    exist_ok=True,
    degrees=5,
    translate=0.1,
    scale=0.4,
    fliplr=0.5,
    mosaic=0.5,
)

BEST = RUN_DIR / "train" / "weights" / "best.pt"
if not BEST.exists():
    raise FileNotFoundError(f"best.pt tidak ditemukan: {BEST}")
print("best model:", BEST)

In [ ]:
import json

best_model = YOLO(str(BEST))
valid_metrics = best_model.val(data=str(DATA_DIR / "data.yaml"), split="val", device=0)
test_metrics = best_model.val(data=str(DATA_DIR / "data.yaml"), split="test", device=0)

def metric_dict(metrics):
    return {
        "mAP50-95": float(metrics.box.map),
        "mAP50": float(metrics.box.map50),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
    }

metrics = {"model": MODEL_NAME, "devices": DEVICES, "images": counts, "valid": metric_dict(valid_metrics), "test": metric_dict(test_metrics)}
print(json.dumps(metrics, indent=2))
Path("/kaggle/working/metrics.json").write_text(json.dumps(metrics, indent=2))

In [ ]:
import shutil

artifacts = Path("/kaggle/working/artifacts")
artifacts.mkdir(exist_ok=True)
shutil.copy2(BEST, artifacts / "best-night-thermal.pt")

onnx_path = Path(best_model.export(format="onnx", imgsz=IMGSZ, dynamic=True))
shutil.copy2(onnx_path, artifacts / "best-night-thermal.onnx")

try:
    engine_path = Path(best_model.export(format="engine", imgsz=IMGSZ, half=True, device=0))
    shutil.copy2(engine_path, artifacts / "best-night-thermal.engine")
except Exception as error:
    print("TensorRT engine export dilewati:", error)

zip_path = shutil.make_archive("/kaggle/working/best-night-thermal", "zip", artifacts)
for file in sorted(artifacts.iterdir()):
    print(file.name, round(file.stat().st_size / 1024 / 1024, 2), "MB")
print("download:", zip_path)

## Setelah selesai

Download `/kaggle/working/best-night-thermal.zip` atau file di folder `artifacts/`, lalu simpan ke `D:/KRTI/model/`.

Catatan: API key hanya dibaca dari Kaggle Secret `ROBOFLOW_API_KEY`; jangan menaruh key langsung di notebook atau commit ke Git. Evaluasi test hanya 66 gambar, jadi cek juga pada rekaman thermal UAV nyata sebelum operasi.